# **Module 06: Inferential Statistics**

---

## 📌 Overview

**Inferential Statistics** is the branch of statistics that allows us to draw conclusions about an **entire population** based on information collected from a **sample**. Since it is rarely feasible or cost-effective to observe every single element of a target population, statistical inference relies on probability theory to estimate unknown parameters, quantify the uncertainty of those estimates, and rigorously test hypotheses.

In this notebook, we will explore the fundamentals of sampling, the Central Limit Theorem, confidence interval construction, and the main hypothesis tests used in Data Science.

---

## 🎯 Learning Objectives

By the end of this module, you will be able to:

* **Understand the relationship between population and sample**, distinguishing population parameters from sample statistics.
* **Apply the Central Limit Theorem (CLT)** to understand the behavior of the sampling distribution of the mean.
* **Construct and interpret Confidence Intervals (CI)** for means and proportions, evaluating margin of error and sample size.
* **Formulate and conduct Hypothesis Tests**, defining null ($H_0$) and alternative ($H_1$) hypotheses, significance level ($\alpha$), and accurately interpreting the **p-value**.
* **Distinguish statistical decision errors** (Type I Error and Type II Error) and understand the concept of statistical power.
* **Perform parametric and non-parametric tests in Python** (such as $t$-test, $Z$-test, ANOVA, Chi-square, and Mann-Whitney) using `scipy.stats` and `statsmodels`.

---

## 🧭 Module Structure

1. **Sampling and Sampling Distributions** — Sampling techniques, sampling error, and the Central Limit Theorem.
2. **Point and Interval Estimation** — Estimators, Confidence Intervals (Normal and Student's $t$ distributions), and sample size determination.
3. **Hypothesis Testing Fundamentals** — $H_0$, $H_1$, critical region, significance level ($\alpha$), p-value, and Type I & Type II Errors.
4. **Parametric Tests** — $Z$-test, $t$-test (one-sample, independent, and paired), and ANOVA.
5. **Non-Parametric and Goodness-of-Fit Tests** — Chi-square ($\chi^2$), Mann-Whitney, Wilcoxon, and Shapiro-Wilk (normality test).
6. **Hands-on Python** — Complete practical application of statistical inference using `scipy.stats`, `statsmodels`, `numpy`, `pandas`, and `seaborn`.

---

## 🛠️ Technical Requirements

* Prior knowledge of **Random Variables and Probability Distributions** (*Module 05*).
* Python environment configured with: `numpy`, `pandas`, `scipy`, `statsmodels`, `matplotlib`, and `seaborn`.

In [1]:
# Import libraries
import pandas as pd
import numpy as np
import math
import random
import matplotlib.pyplot as plt
import seaborn as sns
import statistics as st
import scipy.stats as stats
from scipy.stats import norm

from sklearn.model_selection import train_test_split
from scipy.stats import chi2_contingency, spearmanr

## **Sampling**

* `Population:` the set of all individuals, objects, or elements to be studied. A `census` is the study of every element in the population. A population can be `finite`, when it can be counted (e.g., the number of employees in a company), or `infinite`, when it cannot (e.g., the number of products manufactured by a factory over time).

* `Sample:` a subset drawn from the population, from which we use appropriate statistical procedures to generalize, infer, or draw conclusions about the population.

### **Types of Sampling**
- `Random or probabilistic:` every element in the population has a known (and non-zero) probability of being selected.
    * `Simple`
    * `Systematic`
    * `Stratified`
    * `Cluster`

    * Advantages: rigorous selection without subjectivity, and the ability to mathematically determine sample size and confidence level.

    * Disadvantages: difficulty obtaining a complete list of the population, and random selection can produce a geographically dispersed sample.

- `Non-random or non-probabilistic:` the probability of selection for some or all elements of the population is unknown.
    * `Convenience`
    * `Judgmental (Purposive)`
    * `Quota`
    * `Snowball`

    * Advantages: lower cost, shorter study time, and less labor.

    * Disadvantages: some population units have no chance of being chosen, bias may be introduced, and it is not possible to determine the confidence level of the results.

### **Random Sampling**

#### **Simple Random Sampling**

The simplest and one of the most important sample selection methods.

Design:
1. Use a random procedure to draw elements from the population, all with the same probability.
2. Repeat the process until a sample of $n$ observations is obtained.
3. Each drawn element can be removed from the population (`without replacement`) or returned (`with replacement`), in which case a unit may be selected more than once.

In [2]:
np.random.seed(42)
random.seed(42)

population = {
    'ID': list(range(1, 31)),
    'Value': [6.4, 6.2, 7.0, 6.8, 7.2, 6.4, 6.5, 7.1, 6.8, 6.9, 7.0, 7.1, 6.6, 6.8, 6.7,
              6.3, 6.6, 7.2, 7.0, 6.9, 6.8, 6.7, 6.5, 7.2, 6.8, 6.9, 7.0, 6.7, 6.9, 6.9]
}

df_population = pd.DataFrame(population).set_index("ID")
print(df_population.shape)
df_population.head()

(30, 1)


,Value
ID,
1,6.4
2,6.2
3,7.0
4,6.8
5,7.2


##### *Without Replacement*

In [3]:
# Simple random sampling without replacement from the DataFrame
sample_without_replacement = df_population.sample(n=10)
print(sample_without_replacement.shape)
sample_without_replacement

(10, 1)


,Value
ID,
28,6.7
16,6.3
24,7.2
18,7.2
9,6.8
10,6.9
29,6.9
25,6.8
13,6.6


##### *With Replacement*

In [4]:
# Simple random sampling with replacement from the DataFrame
sample_with_replacement = df_population.sample(n=10, replace=True)
print(sample_with_replacement.shape)
sample_with_replacement

(10, 1)


,Value
ID,
10,6.9
28,6.7
28,6.7
16,6.3
15,6.7
30,6.9
30,6.9
15,6.7
30,6.9


#### **Systematic Random Sampling**

Basically, in systematic sampling, elements are ordered and drawn periodically. *Example: inspecting every 50th part on a production line.*

Design:
1. Determine the sampling interval
$k = N / n$
2. Randomly draw a starting point among the first $k$ elements to introduce randomness.
3. From that point, select every $k$-th element until reaching a sample of size $n$.

In [5]:
# Systematic sampling every 5 elements
k = 5
start = random.randint(0, k - 1)  # Randomly chosen starting point
sample_indices = list(range(start, len(df_population), k))
systematic_sample = df_population.iloc[sample_indices]
systematic_sample

,Value
ID,
1,6.4
6,6.4
11,7.0
16,6.3
21,6.8
26,6.9


#### **Stratified Random Sampling**

This method divides a heterogeneous population into homogeneous subpopulations (strata) and draws a random sample from each stratum. In `uniform` stratified sampling, the same number of elements is drawn from each stratum (recommended when strata have similar sizes). In `proportional` stratified sampling, the number of elements drawn from each stratum is proportional to its size in the population.

In [6]:
# Define value classes
value_classes = {
    'A': (5., 6.5),
    'B': (6.5, 7.),
    'C': (7., 7.5)
}

In [7]:
for cls, (lower_bound, upper_bound) in value_classes.items():
    df_population.loc[(df_population['Value'] > lower_bound) & (df_population['Value'] <= upper_bound), 'Class'] = cls

df_population.head()

,Value,Class
ID,,
1,6.4,A
2,6.2,A
3,7.0,B
4,6.8,B
5,7.2,C


##### *Uniform*

In [8]:
n = 2  # or any other number appropriate for your case
uniform_sample = df_population.groupby('Class').apply(lambda x: x.sample(n))

# Reset index
uniform_sample.reset_index(drop=True, inplace=True)
uniform_sample

/tmp/ipykernel_289449/2738400486.py:5: Pandas4Warning: The inplace keyword in DataFrame.reset_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  uniform_sample.reset_index(drop=True, inplace=True)


,Value
0,6.2
1,6.4
2,7.0
3,6.6
4,7.1
5,7.2


##### *Proportional*

In [9]:
sample_size = 0.3

# Use train_test_split to stratify based on Class
_, proportional_sample = train_test_split(df_population,
                                          stratify=df_population['Class'],
                                          test_size=sample_size,
                                          random_state=42)

proportional_sample

,Value,Class
ID,,
10,6.9,B
20,6.9,B
14,6.8,B
15,6.7,B
7,6.5,A
29,6.9,B
18,7.2,C
16,6.3,A
11,7.0,B


#### **Cluster Random Sampling**

In cluster sampling, the population is divided into groups of elementary units (clusters), and sampling is performed on the groups rather than on individual elements.

Design:
1. Divide the population into $M$ clusters, not necessarily of equal size.
2. Randomly select a sample of $m < M$ clusters.
3. In `one-stage` cluster sampling, all elements of the selected clusters enter the sample. In `two-stage` cluster sampling, a random sample of elements is also drawn within each selected cluster.

In [10]:
# Define the number of clusters
num_clusters = 5

In [11]:
# Define cluster size
cluster_size = len(df_population) // num_clusters

# Assign primary cluster
clusters = list(range(1, num_clusters + 1))
random.shuffle(clusters)
clusters *= len(df_population) // num_clusters + 1
df_population['PrimaryCluster'] = clusters[:len(df_population)]

# Assign secondary cluster
clusters = list(range(1, num_clusters + 1))
random.shuffle(clusters)
clusters *= len(df_population) // num_clusters + 1
df_population['SecondaryCluster'] = clusters[:len(df_population)]

In [12]:
print(df_population.shape)
df_population.head()

(30, 4)


,Value,Class,PrimaryCluster,SecondaryCluster
ID,,,,
1,6.4,A,2,5
2,6.2,A,4,4
3,7.0,B,5,3
4,6.8,B,3,1
5,7.2,C,1,2


##### *One-Stage*

In [13]:
# Primary clusters
primary_clusters = df_population['PrimaryCluster'].unique()

# Define the number of primary clusters to select
num_primary_clusters_sample = 2

# Randomly select primary clusters
primary_clusters_sample = random.sample(list(primary_clusters), k=num_primary_clusters_sample)

# Perform one-stage cluster sampling
one_stage_sample = df_population[df_population['PrimaryCluster'].isin(primary_clusters_sample)]
print(one_stage_sample.shape)
one_stage_sample

(12, 4)


,Value,Class,PrimaryCluster,SecondaryCluster
ID,,,,
4,6.8,B,3,1
5,7.2,C,1,2
9,6.8,B,3,1
10,6.9,B,1,2
14,6.8,B,3,1
15,6.7,B,1,2
19,7.0,B,3,1
20,6.9,B,1,2
24,7.2,C,3,1


##### *Two-Stage*

In [14]:
# Secondary clusters
secondary_clusters = one_stage_sample['SecondaryCluster'].unique()

# Define the number of secondary clusters to select
num_secondary_clusters_sample = 1

# Randomly select secondary clusters
secondary_clusters_sample = random.sample(list(secondary_clusters), k=num_secondary_clusters_sample)

# Perform two-stage cluster sampling
two_stage_sample = one_stage_sample[one_stage_sample['SecondaryCluster'].isin(secondary_clusters_sample)]
print(two_stage_sample.shape)
two_stage_sample

(6, 4)


,Value,Class,PrimaryCluster,SecondaryCluster
ID,,,,
4,6.8,B,3,1
9,6.8,B,3,1
14,6.8,B,3,1
19,7.0,B,3,1
24,7.2,C,3,1
29,6.9,B,3,1


### **Non-Random Sampling**

#### **Convenience Sampling**

Used when participation is voluntary or when elements are chosen out of convenience or simplicity, such as friends, neighbors, and students.

In [15]:
convenience_ids = [5, 10, 15, 20, 25]
convenience_sample = df_population[df_population.index.isin(convenience_ids)]
convenience_sample

,Value,Class,PrimaryCluster,SecondaryCluster
ID,,,,
5,7.2,C,1,2
10,6.9,B,1,2
15,6.7,B,1,2
20,6.9,B,1,2
25,6.8,B,1,2


#### **Judgmental (Purposive) Sampling**

The sample is intentionally chosen based on an expert's judgment or a business rule, which can lead to mistaken conclusions and bias the analysis. *Using this technique requires thorough knowledge of the population and the selected elements.*

*Example: a small company chooses to interview a specific customer because they fit the customer profile the company wants to strengthen.*

In [16]:
judgmental_sample = df_population[(df_population['Value'] >= 6) & (df_population['Value'] <= 6.5)]
judgmental_sample

,Value,Class,PrimaryCluster,SecondaryCluster
ID,,,,
1,6.4,A,2,5
2,6.2,A,4,4
6,6.4,A,2,5
7,6.5,A,4,4
16,6.3,A,2,5
23,6.5,A,5,3


#### **Quota Sampling**

Offers greater rigor than other non-random sampling methods and is widely used in market research and political polling. It can be viewed as a non-probabilistic version of stratified sampling.

Procedure:
1. Select the control variables—that is, the relevant characteristics of the population.
2. Determine the population proportion in each category of those variables.
3. Size the quotas (number of people to be interviewed in each category) so that the sample proportions match those of the population.

In [17]:
# Define the sample size for each class
n = 1

# Quota sample
quota_sample = df_population.groupby('Class').apply(lambda x: x.sample(min(len(x), n)))

# Reset index
quota_sample.reset_index(drop=True, inplace=True)
quota_sample

/tmp/ipykernel_289449/1297478343.py:8: Pandas4Warning: The inplace keyword in DataFrame.reset_index is deprecated and will be removed in a future version (PDEP-8).
See https://pandas.pydata.org/docs/dev/whatsnew/v3.1.0.html#deprecation-inplace for more details.
  quota_sample.reset_index(drop=True, inplace=True)


,Value,PrimaryCluster,SecondaryCluster
0,6.3,2,5
1,7.0,2,5
2,7.1,4,4


#### **Snowball Sampling (Chain-Referral Sampling)**
Widely used when the target population is rare, hard to reach, or even unknown—for example, a recruiting firm looking for professionals with a very specific profile. It is common in social research. Essentially, we identify one or more individuals from the target population, who then refer others, repeating the process until the desired sample size is reached.

In [18]:
# Adding three rare elements to the population
df_population.loc[31] = [10, "Z", 0, 0]
df_population.loc[32] = [8, "Z", 0, 0]
df_population.loc[33] = [9.3, "Z", 0, 0]

In [19]:
def snowball_sampling(df, initial_sample_size, iterations, col):
    # Select an initial random sample
    initial_sample = df.sample(initial_sample_size)

    for i in range(iterations):
        # Find unique values in the target column of our current sample
        unique_values = initial_sample[col].unique()

        # Select specific elements (apply business rule or analysis criteria)
        new_samples = df[df[col] > 7]

        # Add new samples to the initial sample set
        initial_sample = pd.concat([initial_sample, new_samples])

        # Remove duplicates
        initial_sample = initial_sample.drop_duplicates()

    return initial_sample

In [20]:
sample = snowball_sampling(df_population, initial_sample_size=5, iterations=100, col="Value")
sample

,Value,Class,PrimaryCluster,SecondaryCluster
ID,,,,
19,7.0,B,3,1
10,6.9,B,1,2
14,6.8,B,3,1
6,6.4,A,2,5
21,6.8,B,2,5
5,7.2,C,1,2
8,7.1,C,5,3
12,7.1,C,4,4
18,7.2,C,5,3


## **Sample Size**

**Factors that determine sample size:**

1. Population characteristics, such as variance $\sigma^2$ and size $N$.
2. Sampling distribution of the estimator.
3. Required precision and confidence, specifying the margin of error $B$ (maximum difference the researcher accepts between the population parameter and the estimate).
4. Cost: the larger the sample, the higher the cost of the operation.
5. Cost vs. error: larger samples reduce sampling error but increase cost, so both must be balanced.
6. Statistical techniques to be used, as some require larger samples than others.

#### **Simple Random Sample Size**

Calculating sample size to estimate the mean and proportion, in finite and infinite populations, with confidence level $1 - \alpha$.

`Mean`

With margin of error $B$, population mean $\mu$, and sample mean $\overline X$, we want $|\mu - \overline X| \leq B$:

* Infinite population: $n = \frac{z^2_{\alpha/2}\cdot\sigma^2}{B^2}$
* Finite population: $n = \frac{N\cdot z^2_{\alpha/2}\cdot\sigma^2}{(N-1)\cdot B^2 + z^2_{\alpha/2}\cdot\sigma^2}$

`Proportion`

With margin of error $B$, population proportion $p$, and sample proportion $\hat p$, we want $|p - \hat p| \leq B$:

* Infinite population: $n = \frac{z^2_{\alpha/2}\cdot p(1-p)}{B^2}$
* Finite population: $n = \frac{N\cdot z^2_{\alpha/2}\cdot p(1-p)}{(N-1)\cdot B^2 + z^2_{\alpha/2}\cdot p(1-p)}$

`NOTE:` *When $p$ is unknown, $p = 0.5$ is used, which yields the largest (most conservative) sample size.*

In [21]:
random.seed(42)

data = {'Value': [],
        'BinaryValue': []}

for _ in range(10000):  # Generating a population of 10,000 elements
    value = random.uniform(3.0, 10.0)  # Generate random values between 3.0 and 10.0
    data['Value'].append(value)

    binary_val = random.randint(0, 1)
    data['BinaryValue'].append(binary_val)

df = pd.DataFrame(data)

In [22]:
def sample_size_mean(population, error, confidence=0.95, finite=False):

    z = norm.ppf(1 - (1 - confidence) / 2)  # Critical value for the confidence level
    std_dev = population.std()  # Population standard deviation
    N = population.size

    if finite:
        size = (N * (z**2) * (std_dev**2)) / ((N - 1) * (error**2) + (z**2) * (std_dev**2))
    else:
        size = ((z**2) * (std_dev**2)) / (error**2)

    return math.ceil(size)

def sample_size_proportion(population, proportion, error, confidence=0.95, finite=False):

    z = norm.ppf(1 - (1 - confidence) / 2)  # Critical value for the confidence level
    N = population.size

    if finite:
        size = (N * (z**2) * proportion * (1 - proportion)) / ((N - 1) * (error**2) + (z**2) * proportion * (1 - proportion))
    else:
        size = ((z**2) * proportion * (1 - proportion)) / (error**2)

    return math.ceil(size)

In [23]:
error = 0.05
proportion = 0.5

# For a finite population
sample_size_mean_finite = sample_size_mean(df["Value"], error, finite=True)
sample_size_prop_finite = sample_size_proportion(df["Value"], proportion, error, finite=True)

print(f"Sample size MEAN (finite population): {sample_size_mean_finite}")
print(f"Sample size PROPORTION (finite population): {sample_size_prop_finite}")

Sample size MEAN (finite population): 3876
Sample size PROPORTION (finite population): 370


In [24]:
error = 0.05
proportion = 0.5

# For an infinite population
sample_size_mean_infinite = sample_size_mean(df["Value"], error)
sample_size_prop_infinite = sample_size_proportion(df["Value"], proportion, error)

print(f"Sample size MEAN (infinite population): {sample_size_mean_infinite}")
print(f"Sample size PROPORTION (infinite population): {sample_size_prop_infinite}")

Sample size MEAN (infinite population): 6328
Sample size PROPORTION (infinite population): 385


#### **Systematic Sample Size**

Systematic sampling uses the same formulas as simple random sampling, choosing the mean formula (quantitative variable) or the proportion formula (binary variable), along with the finite or infinite population version.

#### **Stratified Sample Size**

In this case, the calculation considers the variance (or proportion) of each stratum: we use the mean formula when the variable is quantitative and the proportion formula when it is binary, following the same logic for finite and infinite populations.

In [25]:
random.seed(42)

# Create DataFrame
data = {'Field': [], 'Value': [], 'Class': []}

for _ in range(5285):
    data['Field'].append('Exact Sciences')
    data['Value'].append(random.uniform(3.0, 50.0))
    data['Class'].append(1)

for _ in range(3877):
    data['Field'].append('Humanities')
    data['Value'].append(random.uniform(3.0, 50.0))
    data['Class'].append(1)

for _ in range(27724):
    data['Field'].append('Biological Sciences')
    data['Value'].append(random.uniform(3.0, 50.0))
    data['Class'].append(0)

df = pd.DataFrame(data)

In [26]:
def stratified_sample_size_mean(df, stratum_col, quant_col, error, confidence=0.95, finite=False):

    z = norm.ppf(1 - (1 - confidence) / 2)  # Critical value for the confidence level
    total_size = 0

    for group in df[stratum_col].unique():
        group_pop = df[df[stratum_col] == group]
        std_dev = group_pop[quant_col].std()  # Stratum standard deviation
        N = len(group_pop)

        if finite:
            group_size = (N * (z**2) * (std_dev**2)) / ((N - 1) * (error**2) + (z**2) * (std_dev**2))
        else:
            group_size = ((z**2) * (std_dev**2)) / (error**2)

        total_size += math.ceil(group_size)

    return total_size


def stratified_sample_size_proportion(data, proportions, total_sample_size, finite=False):

    # Weight of each stratum: expected proportion times stratum size
    weights = {cls: prop * len(data[data['Class'] == cls]) for cls, prop in proportions.items()}
    total_weight = sum(weights.values())

    # Distribute total sample among strata according to weights
    sample_sizes = {}
    for cls, weight in weights.items():
        class_size = len(data[data['Class'] == cls])
        stratum_size = (weight / total_weight) * total_sample_size

        if finite:
            stratum_size = stratum_size / (1 + (stratum_size / class_size))

        sample_sizes[cls] = round(stratum_size)

    return sample_sizes

In [27]:
error = 1.0  # Maximum error of 1 unit in the mean
proportions = {0: 0.5, 1: 0.5}

# For a finite population
sample_size_mean_finite = stratified_sample_size_mean(df, 'Class', 'Value', error, finite=True)
sample_size_prop_finite = stratified_sample_size_proportion(df, proportions, sample_size_mean_finite, finite=True)

print(f"Sample size MEAN (finite population): {sample_size_mean_finite}")
print(f"Sample allocation across strata (finite population): {sample_size_prop_finite}")

Sample size MEAN (finite population): 1339
Sample allocation across strata (finite population): {0: 971, 1: 321}


In [28]:
error = 1.0  # Maximum error of 1 unit in the mean
proportions = {0: 0.5, 1: 0.5}

# For an infinite population
sample_size_mean_infinite = stratified_sample_size_mean(df, 'Class', 'Value', error)
sample_size_prop_infinite = stratified_sample_size_proportion(df, proportions, sample_size_mean_infinite)

print(f"Sample size MEAN (infinite population): {sample_size_mean_infinite}")
print(f"Sample allocation across strata (infinite population): {sample_size_prop_infinite}")

Sample size MEAN (infinite population): 1407
Sample allocation across strata (infinite population): {0: 1058, 1: 349}


#### **Cluster Sample Size**

Here, the sampled unit is the cluster. In `one-stage` sampling, we calculate how many clusters $m$ to draw using the variance between cluster means $S_c^2$. In `two-stage` sampling, we also calculate how many elements to draw within each cluster using the average within-cluster variance $S_w^2$.

$m = \frac{M\cdot z^2_{\alpha/2}\cdot S_c^2}{(M-1)\cdot B^2 + z^2_{\alpha/2}\cdot S_c^2}$ (finite population of $M$ clusters)

`NOTE:` *These expressions are a simplified version. The reference textbook presents the complete formulas combining both variances.*

##### *One-Stage*

In [29]:
def sample_size_1_stage(df, sampling_error, column, cluster_col, confidence=0.95, finite=False):

    z = norm.ppf(1 - (1 - confidence) / 2)  # Critical value for the confidence level
    means = df.groupby(cluster_col)[column].mean()
    M = len(means)  # Number of clusters in the population
    between_var = means.var()  # Variance between cluster means

    if finite:
        m = (M * (z**2) * between_var) / ((M - 1) * (sampling_error**2) + (z**2) * between_var)
    else:
        m = ((z**2) * between_var) / (sampling_error**2)

    return math.ceil(m)

In [30]:
# Parameters for calculating the ideal sample size
sampling_error = 0.5

# Number of clusters to select (one-stage method)
sample_size_1_infinite = sample_size_1_stage(df, sampling_error, 'Value', 'Field')
sample_size_1_finite = sample_size_1_stage(df, sampling_error, 'Value', 'Field', finite=True)

print(f"Clusters to select (1-stage, infinite population): {sample_size_1_infinite}")
print(f"Clusters to select (1-stage, finite population): {sample_size_1_finite}")

Clusters to select (1-stage, infinite population): 1
Clusters to select (1-stage, finite population): 1


##### *Two-Stage*

In [31]:
def sample_size_2_stages(df, sampling_error, column, cluster_col, confidence=0.95, finite=False):

    z = norm.ppf(1 - (1 - confidence) / 2)  # Critical value for the confidence level

    # 1st stage: number of clusters
    m = sample_size_1_stage(df, sampling_error, column, cluster_col, confidence, finite)

    # 2nd stage: elements within each cluster
    within_var = df.groupby(cluster_col)[column].var().mean()  # Average variance within clusters
    avg_size = df.groupby(cluster_col).size().mean()  # Average cluster size

    if finite:
        n = (avg_size * (z**2) * within_var) / ((avg_size - 1) * (sampling_error**2) + (z**2) * within_var)
    else:
        n = ((z**2) * within_var) / (sampling_error**2)

    return m, math.ceil(n)

In [32]:
# Parameters for calculating the ideal sample size
sampling_error = 0.5

# Clusters and elements per cluster (two-stage method)
m_inf, n_inf = sample_size_2_stages(df, sampling_error, 'Value', 'Field')
m_fin, n_fin = sample_size_2_stages(df, sampling_error, 'Value', 'Field', finite=True)

print(f"2-stage, infinite population: {m_inf} cluster(s) with {n_inf} elements each")
print(f"2-stage, finite population: {m_fin} cluster(s) with {n_fin} elements each")

2-stage, infinite population: 1 cluster(s) with 2815 elements each
2-stage, finite population: 1 cluster(s) with 2291 elements each


### **Exercise 11**

Calculate the sample size for each region, with a 99% confidence level and a sampling error of 0.6, given the number of inhabitants and variance for each region.

In [33]:
def stratified_sample_size(n_inhabitants, variance, sampling_error, confidence=0.99):
    z = norm.ppf(1 - (1 - confidence) / 2)  # Critical value (2.576 for 99%)
    sample_size = ((z**2) * variance * n_inhabitants) / ((sampling_error**2) * (n_inhabitants - 1) + (z**2) * variance)
    return math.ceil(sample_size)

In [34]:
# Regional data
data = {'Region': [], 'Inhabitants': [], 'Variance': []}

regions = ['North', 'South', 'Central', 'East', 'West']
region_ids = [1, 2, 3, 4, 5]
inhabitants = [14060, 19477, 36564, 26424, 23475]
variances = [44.5, 59.3, 82.4, 66.2, 69.5]

data["Region"] = regions
data["RegionID"] = region_ids
data["Inhabitants"] = inhabitants
data["Variance"] = variances

df = pd.DataFrame(data)
df

,Region,Inhabitants,Variance,RegionID
0,North,14060,44.5,1
1,South,19477,59.3,2
2,Central,36564,82.4,3
3,East,26424,66.2,4
4,West,23475,69.5,5


In [35]:
# Parameters for sample size calculation
sampling_error = 0.6
confidence = 0.99

# Calculate sample size for each region
sample_sizes = []
for _, row in df.iterrows():
    s_size = stratified_sample_size(row['Inhabitants'], row['Variance'], sampling_error, confidence)
    sample_sizes.append(s_size)

# Display results
for i in range(len(df)):
    print(f"Sample size for {df['Region'][i]} region: {sample_sizes[i]}")

Sample size for North region: 775
Sample size for South region: 1035
Sample size for Central region: 1459
Sample size for East region: 1167
Sample size for West region: 1215
